# Õppetund 13 - Agendi mälu Cognee teadmistegraafikutega


## Seadistamine

See märkmik demonstreerib, kuidas ehitada intelligentne **koodiassistendi** püsiva mäluga, kasutades [**Cognee**](https://www.cognee.ai/) teadmiste graafe ja **Microsoft Agent Frameworki** (MAF).

Cognee teisendab struktureerimata teksti struktureeritud, päringutega juhitavaks teadmiste graafiks, mida toetavad vektori manused — andes teie agendile rikkaliku, suhete-teadliku pikaajalise mälu.

### Mida Sa Õpid
1. **Ehita teadmiste graafid**: Muuda arendajate profiilid ja parimad tavad struktureeritud, päringutega juhitavaks teadmiseks.
2. **Integreeri Cognee MAFiga**: Kasuta `@tool` funktsioone, et lasta MAF agendil pärida Cognee teadmiste graafi.
3. **Seanssiteadlikud vestlused**: Säilita konteksti mitme küsimuse jooksul samas seansis.
4. **Pikaajaline mälu**: Säilita olulist teadmist seansside vahel ja too see uutesse vestlustesse tagasi.

### Eeltingimused
- Python 3.9+
- Kohalikult töötav Redis (`docker run -d -p 6379:6379 redis`) seansihalduseks
- LLM API võti (nt OpenAI) — seadista `LLM_API_KEY` failis `.env`
- `CACHING=true` failis `.env` (nõutud Cognee seansside jaoks)
- Azure AI Foundry projekt koos juurutatud vestlusmudeliga
- `AZURE_AI_PROJECT_ENDPOINT` ja `AZURE_AI_MODEL_DEPLOYMENT_NAME` failis `.env`
- Azure CLI sisse logitud (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Agendi mälu tüübid

See märkmik uurib samu kolme mälu tüüpi nagu põhiosas 13. õppetüki märkmikus, kuid kasutab Cogneet pikaajalise mälu tagapõhjana:

| Mälu tüüp | Mehanism | Eluiga |
|---|---|---|
| **Töömälu** | `agent.create_session()` (MAF) | Üks vestlus |
| **Lühiajaline** | Cognee sessiooni vahemälu (Redis) | Üks sessioon |
| **Pikaajaline** | Cognee teadmiste graafik + vektorid | Püsiv |

### Cognee mäluarhitektuur
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Valmista ette Cognee salvestusruum


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Osa 1 — Teadmusbaasi loomine

Me kogume kolme tüüpi andmeid, et luua põhjalik teadmusbaas meie programmeerimisabilisele:

1. **Arendaja profiil** — isiklik asjatundlikkus ja tehniline taust  
2. **Python'i parimad tavad** — Python'i Zen koos praktiliste juhistega  
3. **Ajaloolised vestlused** — varasemad arendajate ja tehisintellekti abiliste vahelised küsimused ja vastused


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Teadmiste graafiku visualiseerimine

Cognee suudab kuvada interaktiivset HTML-visualisatsiooni tuvastatud üksustest ja nendevahelistest suhetest.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Täienda mälu Memify abil

`memify()` analüüsib teadmiste graafikut ja genereerib intelligentseid reegleid — tuvastades mustreid, parimaid tavasid ja kontseptsioonide vahelisi seoseid.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Osa 2 — MAF-agent Cognee tööriistadega

Nüüd loome MAF-agendi, kes saab pärida Cognee teadmistegraafikut `@tool` funktsioonide kaudu. See võimaldab agendil kasutada graafiteadlikku semantilist otsingut kogu selle võimsuses, säilitades samal ajal vestluse konteksti sessioonide kaudu.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Töömälu sessioonidega

`AgentSession` (loodud funktsiooniga `agent.create_session()`) pakub töömälu sessiooni jooksul. Agent saab viidata varasematele sõnumitele ning samal ajal pärida Cognee pikaajalist teadmiste graafi.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Uus seanss — pikaajaline mälu säilib

Uue seansi alustamine puhastab töömälusse, kuid Cognee teadmiste graaf on endiselt saadaval. Agent saab sama pikaajalist teadmist täiesti uues vestluses taastada.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Kokkuvõte

Selles märkmikus ehitasite kodeerimisassistendi, mis ühendab **MAF töömälu** (`agent.create_session()`) ja **Cogne ei pikaajalise teadmiste graafi**.

### Mida sa õppisid
1. **Teadmiste graafi ülesehitus**: Cognee töötleb struktureerimata teksti ja loob graafi + vektormälu.
2. **Memify-ga graafi rikastamine**: Järeldatud faktid ja rikkalikumad seosed olemasoleva graafi peal.
3. **MAF + Cognee integratsioon**: `@tool` funktsioonid võimaldavad MAF agendil Cognee graafi loomulikult pärida.
4. **Töömälu + pikaajaline mälu**: `AgentSession` (läbi `agent.create_session()`) annab sessioonikonteksti, samal ajal kui Cognee pakub püsivat teadmist.
5. **Filtreeritud otsing NodeSets-iga**: Sihtida teadmiste graafi konkreetseid alamhulki (nt ainult põhimõtteid).

### Peamised järeldused
- **Cognee** muudab toorteksti struktureeritud, seostele orienteeritud mäluks — võimsam kui lihtne vektoripood.
- **`@tool` funktsioonid** loovad puhta silla MAF agentide ja välistingimusega teadmistesüsteemide vahel.
- **`AgentSession`** (läbi `agent.create_session()`) hoiab vestluspõhise konteksti eraldi pikaealisest teadmistest.
- Sama teadmiste graaf teenindab mitut sessiooni ja agenti.

### Reaalsed rakendused
- **Arendaja kaaslased**: Koodi ülevaatus, intsidentide analüüs, arhitektuuri assistendid
- **Kliendisuhtluse kaaslased**: Toetuse esindajad tootetarkvara dokumentide, KKK ja CRM märkmete põhjal
- **Sisemised ekspertide kaaslased**: Poliitika, õigus- või turvalisuse assistendid, kes töötlevad juhiseid
- **Ühtsed andmekihid**: Kombineerida struktureeritud ja struktureerimata andmed üheks päringute graafiks

### Järgmised sammud
- Katsetada ajateadlikkust Cogne es
- Määratleda OWL ontoloogia domeenispetsiifilise graafi kvaliteedi jaoks
- Lisada kasutajate tagasiside tsüklid päringutulemuste ajapikku parandamiseks
- Laiendada multi-agent süsteemideks, mis jagavad sama Cognee mälu kihti


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:
See dokument on tõlgitud tehisintellektil põhineva tõlketeenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi me pingutame täpsuse nimel, palun arvestage, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste või valesti mõistmiste eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
